# StudyBot 연결 설정

대상: **cjdwneo2021-netizen/sys**.

Google Cloud Console에서 프로젝트를 먼저 생성하세요. 아래 셀은 실행 시 전용 서비스 계정, GitHub OIDC 연결과 API 활성화를 구성합니다. private key는 생성하지 않습니다. Drive 접근은 이후 학습 폴더 공유로 설정합니다.


In [ ]:
PROJECT_ID = ""  # Google Cloud 프로젝트 ID만 입력
REPOSITORY_ID = "1406542384"
OWNER_ID = "326789085"
DEFAULT_BRANCH = "main"


In [ ]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)


다음 셀을 실행하면 지정한 프로젝트에 인증 리소스를 생성합니다. 프로젝트의 API 활성화 및 IAM 관리 권한이 필요합니다. 출력은 비밀이 아닌 설정값입니다.


In [ ]:
"""Run in Colab after google.colab.auth.authenticate_user(project_id=...).
Creates a dedicated service account and repository-restricted WIF provider.
No private key is generated.
"""
import json
import re
import subprocess

def gcloud(*args, json_result=False, optional=False):
    command = ["gcloud", *args, "--quiet"]
    if json_result:
        command.append("--format=json")
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode:
        if optional and ("NOT_FOUND" in result.stderr or "not found" in result.stderr.lower()):
            return None
        raise RuntimeError(result.stderr.strip())
    return json.loads(result.stdout) if json_result and result.stdout.strip() else result.stdout.strip()

def setup(project_id, repository_id, owner_id, branch="main"):
    if not re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", project_id):
        raise ValueError("Enter an existing Google Cloud project ID")
    if not str(repository_id).isdigit() or not str(owner_id).isdigit():
        raise ValueError("Repository and owner IDs must be numeric")
    if not re.fullmatch(r"[A-Za-z0-9._/-]+", branch):
        raise ValueError("Invalid branch name")
    project = gcloud("projects", "describe", project_id, json_result=True)
    number = str(project["projectNumber"])
    gcloud("services", "enable", "drive.googleapis.com", "iam.googleapis.com",
           "iamcredentials.googleapis.com", "sts.googleapis.com", "--project", project_id)
    account_id = "studybot-drive"
    account = account_id + "@" + project_id + ".iam.gserviceaccount.com"
    if gcloud("iam", "service-accounts", "describe", account, "--project", project_id,
              json_result=True, optional=True) is None:
        gcloud("iam", "service-accounts", "create", account_id, "--display-name", "StudyBot Drive reader",
               "--project", project_id)
    pool_id = "studybot"
    pool = f"projects/{number}/locations/global/workloadIdentityPools/{pool_id}"
    existing_pool = gcloud("iam", "workload-identity-pools", "describe", pool_id, "--location", "global",
                          "--project", project_id, json_result=True, optional=True)
    if existing_pool is None:
        gcloud("iam", "workload-identity-pools", "create", pool_id, "--location", "global",
               "--display-name", "StudyBot GitHub", "--project", project_id)
    elif existing_pool.get("state") != "ACTIVE" or existing_pool.get("disabled"):
        raise ValueError("Existing Workload Identity Pool is not active")
    mapping = {"google.subject": "assertion.sub", "attribute.repository_id": "assertion.repository_id",
               "attribute.repository_owner_id": "assertion.repository_owner_id", "attribute.ref": "assertion.ref"}
    condition = (f"assertion.repository_id == '{repository_id}' && assertion.repository_owner_id == '{owner_id}'"
                 f" && assertion.ref == 'refs/heads/{branch}'")
    provider_id = "repo-" + str(repository_id)
    existing = gcloud("iam", "workload-identity-pools", "providers", "describe", provider_id,
                      "--workload-identity-pool", pool_id, "--location", "global",
                      "--project", project_id, json_result=True, optional=True)
    if existing is None:
        gcloud("iam", "workload-identity-pools", "providers", "create-oidc", provider_id,
               "--workload-identity-pool", pool_id, "--location", "global", "--project", project_id,
               "--issuer-uri", "https://token.actions.githubusercontent.com",
               "--attribute-mapping", ",".join(key + "=" + value for key, value in mapping.items()),
               "--attribute-condition", condition)
    elif (existing.get("attributeCondition") != condition or existing.get("attributeMapping") != mapping
          or existing.get("oidc", {}).get("issuerUri") != "https://token.actions.githubusercontent.com"
          or existing.get("disabled") or existing.get("state") != "ACTIVE"):
        raise ValueError("Existing provider differs from expected configuration; inspect before reuse")
    member = "principalSet://iam.googleapis.com/" + pool + "/attribute.repository_id/" + str(repository_id)
    gcloud("iam", "service-accounts", "add-iam-policy-binding", account,
           "--project", project_id, "--role", "roles/iam.workloadIdentityUser", "--member", member,
           "--condition=None")
    return {"GCP_WORKLOAD_IDENTITY_PROVIDER": pool + "/providers/" + provider_id,
            "GCP_SERVICE_ACCOUNT": account}

connection = setup(PROJECT_ID, REPOSITORY_ID, OWNER_ID, DEFAULT_BRANCH)
print(json.dumps(connection, indent=2))


위 값을 [GitHub Actions Variables](https://github.com/cjdwneo2021-netizen/sys/settings/variables/actions)에 각각 등록합니다.

1. Drive에서 **StudyBotInbox** 폴더를 만들고 위 `GCP_SERVICE_ACCOUNT` 이메일을 **뷰어**로 추가합니다.
2. 폴더 URL의 `/folders/` 뒤 ID를 Actions Variable `DRIVE_FOLDER_ID`로 저장합니다.
3. 사용할 Gemini 모델의 API 키를 [Actions Secrets](https://github.com/cjdwneo2021-netizen/sys/settings/secrets/actions)에 `GEMINI_API_KEY`로 저장합니다.
4. Variables에 `MODEL_PROVIDER=gemini`, `MODEL_NAME=실제 사용 가능한 모델 ID`를 저장합니다.
5. IAM 설정이 반영되도록 약 5분 기다린 뒤 `STUDYBOT_ENABLED=true`를 등록합니다.

무료 사용 조건·모델별 한도는 Google AI Studio에서 확인하세요. 키나 토큰을 이 노트북에 붙여 넣지 않습니다. 현재 저장소가 Public이면 수집한 학습 자료도 공개됩니다. 비공개 자료라면 자동화 활성화 전에 저장소를 Private으로 바꾸세요.
